# 15. Scaled Agent Memory: Vector vs Bi-Temporal Graph vs Full Context on All of LongMemEval and LoCoMo

**Presentation map:** Slide 52 (memory tiers: episodic → semantic), Slide 53 (bi-temporal edges — a contradiction
*invalidates* the old edge instead of deleting it), Slide 54 (Graphiti-style `add_episode`), Slide 55 (Zep/Graphiti on
LongMemEval), Slide 56 (no memory design wins every question type).

Notebook 10 compared the memories on a stratified sample of 108 LongMemEval questions; several per-type differences
(temporal reasoning, single-session-assistant) were too close to call, and the full-context baseline ran on only 28
questions. This notebook removes those limits:

| Benchmark | Questions | History per question | Source |
|---|---|---|---|
| **LongMemEval-S** (cleaned) | **all 500** (6 types + 30 abstention) | ≈48 sessions, ≈115k tokens, each question its own haystack | Wu et al., ICLR 2025; `xiaowu0162/longmemeval-cleaned`, MIT |
| **LoCoMo** | **1,540** (all except the adversarial category) | 10 long conversations between two people, 19–32 sessions each | Maharana et al., ACL 2024; `snap-research/locomo`, CC BY-NC 4.0 |

**Three memories, one answer model and prompt** (notebook 10's design, unchanged):

| Memory | Index | What the answerer sees |
|---|---|---|
| (a) **Vector memory** | every round (user message + reply; LoCoMo: every turn) embedded with its date | top-10 rounds |
| (b) **Bi-temporal graph memory** | the LLM extracts facts per session; entity resolution; *state* facts invalidate older ones (`invalid_at`), never deleted; provenance on every edge | top-25 facts with validity dates + the source turns of the top-5 |
| (c) **Full context** | none | the whole history (fits in the model's 1M-token window) — on **every** question this time |

**Scoring:** LongMemEval's official type-specific judge prompts (copied from `src/evaluation/evaluate_qa.py`); for
LoCoMo a generous correct/wrong judge in the style used by Mem0 and Zep. *Deviation:* the judge is `gpt-6-luna`.

**Model:** `gpt-6-luna`, no reasoning, via the OpenAI Batch API (`smash_batch.py`). Retrieval results are cached as
JSONL in `data/benchmarks/longmemeval/cache_luna/` and `data/benchmarks/locomo/cache/` (vectors are not stored).

In [1]:
import json, os, re, time, random, hashlib, math
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, Literal
import numpy as np
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot, EMBED_MODEL

MODE = mode_from_env()
TEST = MODE == "test"
LME, LCM = "data/benchmarks/longmemeval", "data/benchmarks/locomo"
llm_a = LLM(f"{LME}/cache_luna" + ("_test" if TEST else ""))
llm_b = LLM(f"{LCM}/cache" + ("_test" if TEST else ""))

def jsonl(path):
    return {r["key"]: r for r in map(json.loads, open(path))} if os.path.exists(path) else {}

def append(path, row):
    with open(path, "a") as f:
        f.write(json.dumps(row) + "\n")

def embed(texts, cap=6000):
    # Uncached embedding (the vectors are large; only retrieval results are cached). Returns (unit vectors, tokens).
    if MODE == "replay":
        raise RuntimeError("retrieval not cached - run with SMASH_MODE=test or batch")
    texts = [(t or " ")[:cap] for t in texts]
    out, used, i = [], 0, 0
    while i < len(texts):
        batch, size = [], 0
        while i < len(texts) and len(batch) < 512 and size + len(texts[i]) < 400_000:
            batch.append(texts[i]); size += len(texts[i]); i += 1
        for attempt in range(12):                     # rate limits: back off up to a minute per try
            try:
                r = llm_a.client.embeddings.create(model=EMBED_MODEL, input=batch); break
            except Exception:
                if attempt == 11:
                    raise
                time.sleep(min(60, 2 ** attempt) + random.random())
        out += [e.embedding for e in r.data]; used += r.usage.total_tokens
    v = np.array(out, np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True), used

def pmap(fn, items, workers=4):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))

# Part A — LongMemEval-S, all 500 questions

In [2]:
data = json.load(open(f"{LME}/longmemeval_s_cleaned.json"))
by_id = {q["question_id"]: q for q in data}
is_abs = lambda q: q["question_id"].endswith("_abs")
TYPES = ["single-session-user", "single-session-assistant", "single-session-preference",
         "multi-session", "knowledge-update", "temporal-reasoning"]
group = lambda qid: "abstention" if is_abs(by_id[qid]) else by_id[qid]["question_type"]
QIDS = sorted(by_id)
if TEST:
    QIDS = [next(q for q in QIDS if group(q) == g) for g in TYPES + ["abstention"]]
print(f"LongMemEval-S: {len(QIDS)} questions |", dict(Counter(map(group, QIDS))))
n_sess = sum(len(by_id[q]["haystack_sessions"]) for q in QIDS)
print(f"{n_sess:,} sessions in the haystacks")

LongMemEval-S: 500 questions | {'single-session-user': 64, 'multi-session': 121, 'knowledge-update': 72, 'abstention': 30, 'single-session-preference': 30, 'temporal-reasoning': 127, 'single-session-assistant': 56}
23,867 sessions in the haystacks


## (a) Vector memory

In [3]:
ROUND_CAP, K_VEC = 1500, 10
VEC_PATH = f"{llm_a.cache}/retrieval_vector.jsonl"
vec_cache = jsonl(VEC_PATH)

def rounds_of(q):
    out = []
    for sid, date, sess in zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]):
        for i in range(0, len(sess), 2):
            txt = "\n".join(f"{t['role'].upper()}: {t['content']}" for t in sess[i:i + 2])
            out.append({"sid": sid, "date": date, "text": f"[{date}] {txt}"})
    return out

def vector_retrieve(qid):
    if qid in vec_cache:
        return vec_cache[qid]
    q = by_id[qid]
    rs = rounds_of(q)
    E, used = embed([r["text"] for r in rs] + [q["question"]])
    top = sorted(np.argsort(-(E[:-1] @ E[-1]))[:K_VEC])
    row = {"key": qid, "context": "\n\n".join(rs[i]["text"][:ROUND_CAP] for i in top), "embed_tokens": used,
           "hit": any(rs[i]["sid"] in q["answer_session_ids"] for i in top)}
    append(VEC_PATH, row)
    return row

VEC = {r["key"]: r for r in pmap(vector_retrieve, QIDS)}
ans_q = [q for q in QIDS if not is_abs(by_id[q])]
print(f"Top-{K_VEC} rounds include an evidence session: {sum(VEC[q]['hit'] for q in ans_q)}/{len(ans_q)}")

Top-10 rounds include an evidence session: 465/470


## (b) Bi-temporal graph memory: one extraction request per unique session (one Batch job)

In [4]:
class Fact(BaseModel):
    subject: str = Field(description="entity the fact is about; use 'user' for the human user")
    relation: str = Field(description="UPPER_SNAKE_CASE relation, e.g. LIVES_IN, OWNS, PLANS_TO, RECOMMENDED")
    object: str = Field(description="value or entity; keep numbers, names and dates exact")
    fact: str = Field(description="one self-contained sentence with all specifics (names, numbers, dates)")
    turn: int = Field(description="index of the turn the fact comes from")
    event_date: Optional[str] = Field(description="YYYY/MM/DD if the fact has its own date (resolve 'last week' against the session date), else null")
    is_state: bool = Field(description="true if this value replaces earlier values of the same subject+relation (current job, address, count, status); false for events and multi-valued facts")

class Episode(BaseModel):
    facts: list[Fact]

EXTRACT_SYS = ("You build the long-term memory of a personal assistant. From one chat session, extract the facts "
               "worth remembering: everything the USER says about themselves (events, possessions, preferences, "
               "plans, people, places, numbers, dates, changes of state) and specific things the ASSISTANT "
               "provided (recommendations, names, numbers, lists the user may ask about later). Skip generic "
               "chit-chat. Be exhaustive for personal details; at most 40 facts.")
TURN_CAP, SESSION_CAP = 4000, 40000
skey = lambda sid, sess: hashlib.sha1((sid + json.dumps(sess)).encode()).hexdigest()

jobs = {}
for qid in QIDS:
    q = by_id[qid]
    for sid, date, sess in zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]):
        jobs.setdefault(skey(sid, sess), (sid, date, sess))
reqs = {}
for k, (sid, date, sess) in jobs.items():
    turns = "\n".join(f"[turn {i}] {t['role'].upper()}: {t['content'][:TURN_CAP]}" for i, t in enumerate(sess))
    reqs[k] = LLM.body([{"role": "system", "content": EXTRACT_SYS},
                        {"role": "user", "content": f"Session date: {date}\n\n{turns[:SESSION_CAP]}"}],
                       max_tokens=6000, schema=Episode)
res = llm_a.run("extractions", reqs, parse=Episode)
EXT = {k: (v["parsed"] or {"facts": []})["facts"] for k, v in res.items()}
print(f"{len(EXT):,} unique sessions -> {sum(map(len, EXT.values())):,} facts "
      f"({sum(v['parsed'] is None for v in res.values())} sessions unparsable, kept without facts)")

19,195 unique sessions -> 122,597 facts (1 sessions unparsable, kept without facts)


In [5]:
SELF = {"i", "me", "my", "myself", "user", "the user", "you (user)", "human"}
def norm(name, self_names=SELF):
    n = re.sub(r"[^a-z0-9 ]+", " ", str(name).lower()).strip()
    n = re.sub(r"^(the|a|an|my) ", "", n)
    return "user" if self_names and (n in self_names or str(name).strip().lower() in self_names) else n

def resolve_and_invalidate(edges, emb, self_names=SELF):
    names = sorted({e["s"] for e in edges} | {norm(e["o"], self_names) for e in edges})
    canon = {n: n for n in names}
    if len(names) > 1:
        V = emb(names)
        for a in range(len(names)):
            for b in np.where((V[a] @ V[a + 1:].T) >= 0.92)[0] + a + 1:
                canon[names[b]] = canon[names[a]]
    for e in edges:
        e["s"] = canon[e["s"]]
    latest, inv = {}, 0
    for e in edges:                                  # edges are in ingestion (chronological) order
        if not e["state"]:
            continue
        k = (e["s"], e["r"]); old = latest.get(k)
        if (old is not None and old["sid"] != e["sid"]
                and canon.get(norm(old["o"], self_names), norm(old["o"], self_names)) != canon.get(norm(e["o"], self_names), norm(e["o"], self_names))):
            old["invalid_at"] = e["valid_at"]; inv += 1
        latest[k] = e
    return sum(1 for n in names if canon[n] != n), inv

def graph_context(edges, question, emb, K_FACTS=25, K_SRC=5, SRC_CAP=1200):
    V = emb([f"{e['s']} {e['r']} {e['o']}: {e['fact']}" for e in edges] + [question])
    sims = V[:-1] @ V[-1]
    qn = norm(question, None)
    sims += np.array([0.05 if len(e["s"]) > 3 and e["s"] != "user" and e["s"] in qn else 0.0 for e in edges])
    top = list(np.argsort(-sims)[:K_FACTS])
    lines = []
    for i in sorted(top, key=lambda i: edges[i]["valid_at"]):
        e = edges[i]
        status = f"superseded {e['invalid_at']}" if e["invalid_at"] else "current"
        lines.append(f"- [{e['valid_at']} | {status}] {e['fact']}  (session {e['sid']}, turn {e['turn']})")
    srcs = [f"[{edges[i]['created_at']} session {edges[i]['sid']} turn {edges[i]['turn']}] {edges[i]['src'][:SRC_CAP]}" for i in top[:K_SRC]]
    return "FACTS FROM MEMORY GRAPH:\n" + "\n".join(lines) + "\n\nSOURCE TURNS OF THE TOP FACTS:\n" + "\n\n".join(srcs), top

GR_PATH = f"{llm_a.cache}/retrieval_graph.jsonl"
gr_cache = jsonl(GR_PATH)
def graph_retrieve(qid):
    if qid in gr_cache:
        return gr_cache[qid]
    q = by_id[qid]
    order = sorted(range(len(q["haystack_dates"])), key=lambda i: q["haystack_dates"][i])
    edges = []
    for i in order:
        sid, date, sess = q["haystack_session_ids"][i], q["haystack_dates"][i], q["haystack_sessions"][i]
        for f in EXT[skey(sid, sess)]:
            t = f["turn"] if isinstance(f["turn"], int) and 0 <= f["turn"] < len(sess) else 0
            edges.append({"s": norm(f["subject"]), "r": f["relation"].upper(), "o": f["object"], "fact": f["fact"],
                          "sid": sid, "turn": t, "created_at": date[:10], "valid_at": f.get("event_date") or date[:10],
                          "invalid_at": None, "state": bool(f["is_state"]), "src": sess[t]["content"]})
    used = [0]
    def emb(texts):
        V, u = embed(texts, cap=2000); used[0] += u; return V
    merged, inv = resolve_and_invalidate(edges, emb)
    ctx, top = graph_context(edges, q["question"], emb)
    row = {"key": qid, "context": ctx, "n_edges": len(edges), "merged": merged, "invalidated": inv,
           "embed_tokens": used[0], "hit": any(edges[i]["sid"] in q["answer_session_ids"] for i in top)}
    append(GR_PATH, row)
    return row

GR = {r["key"]: r for r in pmap(graph_retrieve, QIDS)}
print(f"Graph memories: {np.mean([r['n_edges'] for r in GR.values()]):.0f} edges per history, "
      f"{sum(r['invalidated'] for r in GR.values()):,} edges invalidated (kept), "
      f"top-25 facts include an evidence session: {sum(GR[q]['hit'] for q in ans_q)}/{len(ans_q)}")

Graph memories: 304 edges per history, 2,534 edges invalidated (kept), top-25 facts include an evidence session: 446/470


## Answers (vector, graph and full context for every question) and the official judge

In [6]:
def anscheck_prompt(task, question, answer, response, abstention=False):
    # Copied from LongMemEval src/evaluation/evaluate_qa.py (get_anscheck_prompt).
    if not abstention:
        if task in ['single-session-user', 'single-session-assistant', 'multi-session']:
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'temporal-reasoning':
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. In addition, do not penalize off-by-one errors for the number of days. If the question asks for the number of days/weeks/months, etc., and the model makes off-by-one errors (e.g., predicting 19 days when the answer is 18), the model's response is still correct. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'knowledge-update':
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response contains some previous information along with an updated answer, the response should be considered as correct as long as the updated answer is the required answer.\n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'single-session-preference':
            template = "I will give you a question, a rubric for desired personalized response, and a response from a model. Please answer yes if the response satisfies the desired response. Otherwise, answer no. The model does not need to reflect all the points in the rubric. The response is correct as long as it recalls and utilizes the user's personal information correctly.\n\nQuestion: {}\n\nRubric: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        else:
            raise NotImplementedError
    else:
        template = "I will give you an unanswerable question, an explanation, and a response from a model. Please answer yes if the model correctly identifies the question as unanswerable. The model could say that the information is incomplete, or some other information is given but the asked information is not.\n\nQuestion: {}\n\nExplanation: {}\n\nModel Response: {}\n\nDoes the model correctly identify the question as unanswerable? Answer yes or no only."
    return template.format(question, answer, response)

ANSWER_SYS = ("You are a personal assistant with a memory of your past conversations with the user. Use only the "
              "memory provided. Dates matter: compute elapsed time from the dates shown, and prefer the most recent "
              "value when a fact changed. If the memory does not contain the answer, say so. Think briefly, then "
              "give the final answer on the last line as 'Answer: ...'.")

def full_context(q):
    sess = sorted(zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]), key=lambda x: x[1])
    return "\n\n".join(f"### Session {sid} ({d})\n" + "\n".join(f"{t['role'].upper()}: {t['content']}" for t in s)
                       for sid, d, s in sess)

METHODS = ["vector", "graph", "full"]
reqs = {}
for qid in QIDS:
    q = by_id[qid]
    for m in METHODS:
        ctx = VEC[qid]["context"] if m == "vector" else GR[qid]["context"] if m == "graph" else full_context(q)
        reqs[f"{m}|{qid}"] = LLM.body([{"role": "system", "content": ANSWER_SYS},
                                      {"role": "user", "content": f"Current date: {q['question_date']}\n\nMEMORY:\n{ctx}\n\nQuestion: {q['question']}"}],
                                     max_tokens=800)
ANS_A = llm_a.run("answers", reqs)
jud = llm_a.run("judgments", {k: LLM.body([{"role": "user", "content": anscheck_prompt(
    by_id[k.split("|")[1]]["question_type"], by_id[k.split("|")[1]]["question"], by_id[k.split("|")[1]]["answer"],
    a["text"], abstention=is_abs(by_id[k.split("|")[1]]))}], max_tokens=5) for k, a in ANS_A.items()})
OK_A = {k: "yes" in v["text"].lower() for k, v in jud.items()}
print(f"{len(ANS_A):,} answers, {len(OK_A):,} judgments")

1,500 answers, 1,500 judgments


In [7]:
def acc(ok, keys):
    k = sum(ok[x] for x in keys); lo, hi = wilson(k, len(keys))
    return k / len(keys), lo, hi

RES_A = {}
print(f"{'question group':<27}{'n':>4} | " + " | ".join(f"{m:^18}" for m in METHODS))
for g in TYPES + ["abstention", "ALL"]:
    qs = [q for q in QIDS if g == "ALL" or group(q) == g]
    RES_A[g] = {m: acc(OK_A, [f"{m}|{q}" for q in qs]) for m in METHODS}
    print(f"{g:<27}{len(qs):>4} | " + " | ".join(f"{a:5.0%} [{lo:.0%}-{hi:.0%}]".center(18) for a, lo, hi in RES_A[g].values()))
DIFF_A = {(a, b): paired_boot([OK_A[f"{a}|{q}"] for q in QIDS], [OK_A[f"{b}|{q}"] for q in QIDS])
          for a, b in [("vector", "graph"), ("full", "graph"), ("vector", "full")]}
for (a, b), (dm, lo, hi) in DIFF_A.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")
TOK_A = {m: np.mean([ANS_A[f"{m}|{q}"]["in"] for q in QIDS]) for m in METHODS}
print("Answer-prompt tokens per question: " + ", ".join(f"{m} {t:,.0f}" for m, t in TOK_A.items()))

question group                n |       vector       |       graph        |        full       
single-session-user          64 |    95% [87%-98%]   |    98% [92%-100%]  |    88% [77%-94%]  
single-session-assistant     56 |    91% [81%-96%]   |    52% [39%-64%]   |   100% [94%-100%] 
single-session-preference    30 |    73% [56%-86%]   |    70% [52%-83%]   |    47% [30%-64%]  
multi-session               121 |    60% [51%-68%]   |    76% [68%-83%]   |    57% [48%-65%]  
knowledge-update             72 |    88% [78%-93%]   |    94% [87%-98%]   |    88% [78%-93%]  
temporal-reasoning          127 |    61% [53%-69%]   |    73% [65%-80%]   |    53% [44%-61%]  
abstention                   30 |    97% [83%-99%]   |    90% [74%-97%]   |    97% [83%-99%]  
ALL                         500 |    75% [71%-79%]   |    79% [75%-82%]   |    71% [67%-75%]  
  graph - vector: +3.4% [-0.8%, +7.4%]
  graph - full: +7.8% [+3.0%, +12.6%]
  full - vector: -4.4% [-8.4%, -0.4%]
Answer-prompt tokens per quest

# Part B — LoCoMo: 10 long two-person conversations, 1,540 questions

In [8]:
locomo = json.load(open(f"{LCM}/locomo10.json"))
CAT = {1: "multi-hop", 2: "temporal", 3: "open-domain", 4: "single-hop"}
if TEST:
    locomo = locomo[:1]
convs = {}
for c in locomo:
    conv = c["conversation"]
    sessions = []
    for i in range(1, 100):
        if f"session_{i}" not in conv:
            break
        sessions.append({"sid": f"S{i}", "date": conv[f"session_{i}_date_time"], "turns": conv[f"session_{i}"]})
    qa = [dict(q, qid=f"{c['sample_id']}#{n}") for n, q in enumerate(c["qa"]) if q["category"] in CAT and "answer" in q]
    if TEST:
        qa = [q for cat in CAT for q in [x for x in qa if x["category"] == cat][:4]]
    convs[c["sample_id"]] = {"a": conv["speaker_a"], "b": conv["speaker_b"], "sessions": sessions, "qa": qa}
LQ = [(cid, q) for cid, cv in convs.items() for q in cv["qa"]]
print(f"LoCoMo: {len(convs)} conversations, {sum(len(c['sessions']) for c in convs.values())} sessions, "
      f"{len(LQ):,} questions |", dict(Counter(CAT[q['category']] for _, q in LQ)))

LoCoMo: 10 conversations, 272 sessions, 1,540 questions | {'temporal': 321, 'open-domain': 96, 'multi-hop': 282, 'single-hop': 841}


In [9]:
def turn_text(t, date):
    cap = f" [shares a photo: {t['blip_caption']}]" if t.get("blip_caption") else ""
    return f"[{date}] {t['speaker']}: {t['text']}{cap}"

LVEC_PATH = f"{llm_b.cache}/retrieval_vector.jsonl"
lvec_cache = jsonl(LVEC_PATH)
def lvector(item):
    cid, q = item
    if q["qid"] in lvec_cache:
        return lvec_cache[q["qid"]]
    cv = convs[cid]
    turns = [(s["sid"], t, turn_text(t, s["date"])) for s in cv["sessions"] for t in s["turns"]]
    E, used = embed([x[2] for x in turns] + [q["question"]])
    top = sorted(np.argsort(-(E[:-1] @ E[-1]))[:K_VEC])
    row = {"key": q["qid"], "context": "\n".join(turns[i][2] for i in top), "embed_tokens": used,
           "hit": any(turns[i][1]["dia_id"] in q.get("evidence", []) for i in top)}
    append(LVEC_PATH, row)
    return row
LVEC = {r["key"]: r for r in pmap(lvector, LQ)}

LEXTRACT_SYS = ("You build the long-term memory of an assistant that will answer questions about two people. From one "
                "conversation session between them, extract every fact worth remembering about each person: events "
                "(with dates), possessions, preferences, plans, family, friends, places, numbers, feelings and changes "
                "of state. Use the person's name as subject. Resolve relative dates ('last week', 'yesterday') against "
                "the session date. At most 40 facts.")
reqs = {}
for cid, cv in convs.items():
    for s in cv["sessions"]:
        turns = "\n".join(f"[turn {i}] {t['speaker']}: {t['text']}" + (f" [shares a photo: {t['blip_caption']}]" if t.get("blip_caption") else "")
                          for i, t in enumerate(s["turns"]))
        reqs[f"{cid}|{s['sid']}"] = LLM.body([{"role": "system", "content": LEXTRACT_SYS},
                                              {"role": "user", "content": f"Session date: {s['date']}\nSpeakers: {cv['a']} and {cv['b']}\n\n{turns}"}],
                                             max_tokens=6000, schema=Episode)
lres = llm_b.run("extractions", reqs, parse=Episode)
LEXT = {k: (v["parsed"] or {"facts": []})["facts"] for k, v in lres.items()}
print(f"LoCoMo extraction: {len(LEXT)} sessions -> {sum(map(len, LEXT.values())):,} facts")

from datetime import datetime
def parse_date(s):
    try:
        return datetime.strptime(s.strip(), "%I:%M %p on %d %B, %Y").strftime("%Y/%m/%d")
    except Exception:
        return s

LGR_PATH = f"{llm_b.cache}/retrieval_graph.jsonl"
lgr_cache = jsonl(LGR_PATH)
def lgraph(item):
    cid, q = item
    if q["qid"] in lgr_cache:
        return lgr_cache[q["qid"]]
    cv = convs[cid]
    edges = []
    for s in cv["sessions"]:
        d = parse_date(s["date"])
        for f in LEXT[f"{cid}|{s['sid']}"]:
            t = f["turn"] if isinstance(f["turn"], int) and 0 <= f["turn"] < len(s["turns"]) else 0
            edges.append({"s": norm(f["subject"], None), "r": f["relation"].upper(), "o": f["object"], "fact": f["fact"],
                          "sid": s["sid"], "turn": t, "created_at": d, "valid_at": f.get("event_date") or d,
                          "invalid_at": None, "state": bool(f["is_state"]),
                          "src": f"{s['turns'][t]['speaker']}: {s['turns'][t]['text']}", "dia": s["turns"][t]["dia_id"]})
    used = [0]
    def emb(texts):
        V, u = embed(texts, cap=2000); used[0] += u; return V
    merged, inv = resolve_and_invalidate(edges, emb, self_names=None)
    ctx, top = graph_context(edges, q["question"], emb)
    row = {"key": q["qid"], "context": ctx, "embed_tokens": used[0], "invalidated": inv,
           "hit": any(edges[i]["dia"] in q.get("evidence", []) for i in top)}
    append(LGR_PATH, row)
    return row
LGR = {r["key"]: r for r in pmap(lgraph, LQ)}

def lfull(cv):
    return "\n\n".join(f"### Session {s['sid']} ({s['date']})\n" + "\n".join(turn_text(t, s["date"]).split("] ", 1)[1] for t in s["turns"])
                       for s in cv["sessions"])

LANSWER_SYS = ("You answer questions about two people using only the memory of their past conversations provided. "
               "Dates matter: resolve relative dates against the session dates. Answer concisely; if the memory does not "
               "contain the answer, say so. Give the final answer on the last line as 'Answer: ...'.")
reqs = {}
for cid, q in LQ:
    for m in METHODS:
        ctx = LVEC[q["qid"]]["context"] if m == "vector" else LGR[q["qid"]]["context"] if m == "graph" else lfull(convs[cid])
        reqs[f"{m}|{q['qid']}"] = LLM.body([{"role": "system", "content": LANSWER_SYS},
                                           {"role": "user", "content": f"MEMORY:\n{ctx}\n\nQuestion: {q['question']}"}], max_tokens=400)
ANS_B = llm_b.run("answers", reqs)

class Judgment(BaseModel):
    label: Literal["CORRECT", "WRONG"]

QB = {q["qid"]: q for _, q in LQ}
def ljudge(k, response):
    q = QB[k.split("|", 1)[1]]
    return ("You are grading an answer to a question about a past conversation. Label it CORRECT if it contains the "
            "gold answer or something with the same meaning (be generous with wording; for dates, the same day, month "
            "or year as the gold answer counts, even in another format). Label it WRONG otherwise.\n\n"
            f"Question: {q['question']}\nGold answer: {q['answer']}\nResponse: {response}")
lj = llm_b.run("judgments", {k: LLM.body([{"role": "user", "content": ljudge(k, a["text"])}], max_tokens=20, schema=Judgment)
                             for k, a in ANS_B.items()}, parse=Judgment)
OK_B = {k: (v["parsed"] or {"label": "WRONG"})["label"] == "CORRECT" for k, v in lj.items()}

RES_B = {}
print(f"{'category':<14}{'n':>5} | " + " | ".join(f"{m:^18}" for m in METHODS))
for g in list(CAT.values()) + ["ALL"]:
    qs = [q["qid"] for _, q in LQ if g == "ALL" or CAT[q["category"]] == g]
    RES_B[g] = {m: acc(OK_B, [f"{m}|{x}" for x in qs]) for m in METHODS}
    print(f"{g:<14}{len(qs):>5} | " + " | ".join(f"{a:5.0%} [{lo:.0%}-{hi:.0%}]".center(18) for a, lo, hi in RES_B[g].values()))
lq = [q["qid"] for _, q in LQ]
DIFF_B = {(a, b): paired_boot([OK_B[f"{a}|{x}"] for x in lq], [OK_B[f"{b}|{x}"] for x in lq])
          for a, b in [("vector", "graph"), ("full", "graph"), ("vector", "full")]}
for (a, b), (dm, lo, hi) in DIFF_B.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")
TOK_B = {m: np.mean([ANS_B[f"{m}|{x}"]["in"] for x in lq]) for m in METHODS}
print("Answer-prompt tokens per question: " + ", ".join(f"{m} {t:,.0f}" for m, t in TOK_B.items()))

LoCoMo extraction: 272 sessions -> 3,617 facts


category          n |       vector       |       graph        |        full       
multi-hop       282 |    42% [36%-48%]   |    55% [49%-60%]   |    61% [56%-67%]  
temporal        321 |    69% [64%-74%]   |    79% [75%-83%]   |    75% [70%-80%]  
open-domain      96 |    42% [32%-52%]   |    36% [28%-46%]   |    43% [33%-53%]  
single-hop      841 |    77% [74%-80%]   |    81% [79%-84%]   |    90% [88%-92%]  
ALL            1540 |    67% [64%-69%]   |    73% [71%-75%]   |    79% [77%-81%]  
  graph - vector: +6.6% [+4.2%, +9.0%]
  graph - full: -5.5% [-7.6%, -3.4%]
  full - vector: +12.0% [+9.7%, +14.4%]
Answer-prompt tokens per question: vector 611, graph 1,393, full 20,721


## Cost

In [10]:
print("LongMemEval:"); LA = llm_a.print_ledger()
print("\nLoCoMo:"); LB = llm_b.print_ledger()
emb_tokens = sum(r["embed_tokens"] for r in list(VEC.values()) + list(GR.values()) + list(LVEC.values()) + list(LGR.values()))
print(f"\nRetrieval embeddings (not in the ledgers above): {emb_tokens / 1e6:.1f}M tokens, ${emb_tokens * 0.02 / 1e6:.2f}")

LongMemEval:
stage                           calls     input tok   output tok      USD  via Batch
answers                         1,500    54,813,113       73,666     2.76       100%
extractions                    19,195    50,170,160    9,393,831     4.86       100%
judgments                       1,500       291,252        6,000     0.02       100%
(embeddings)                        0             0            0     0.00         0%
TOTAL                          22,195   105,274,525    9,473,497     7.63

LoCoMo:
stage                           calls     input tok   output tok      USD  via Batch
answers                         4,620    34,997,349      133,500     1.78       100%
extractions                       272       337,705      207,323     0.07       100%
judgments                       4,620       747,140       60,060     0.05       100%
(embeddings)                        0             0            0     0.00         0%
TOTAL                           9,512    36,082,194   

In [11]:
print("=" * 96)
print(f"RESULTS SUMMARY — Notebook 15: agent memory, gpt-6-luna (no reasoning), {MODE} run")
print("=" * 96)
for name, RES, DIFF, TOK, n in (("LongMemEval-S", RES_A, DIFF_A, TOK_A, len(QIDS)), ("LoCoMo", RES_B, DIFF_B, TOK_B, len(LQ))):
    print(f"\n{name} (n = {n}): " + " | ".join(f"{m} {RES['ALL'][m][0]:.1%} [{RES['ALL'][m][1]:.0%}-{RES['ALL'][m][2]:.0%}]" for m in METHODS))
    for (a, b), (dm, lo, hi) in DIFF.items():
        print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  resolvable" if lo > 0 or hi < 0 else "  not resolvable"))
    print("  answer-prompt tokens per question: " + ", ".join(f"{m} {t:,.0f}" for m, t in TOK.items()))
    best = {g: max(METHODS, key=lambda m: RES[g][m][0]) for g in RES if g != "ALL"}
    print("  best memory per question type: " + "; ".join(f"{g}: {m}" for g, m in best.items()))
print("\nLecture message (Slides 52-56): compare the per-type rows — the best memory changes with the question type,"
      "\nand full context is now affordable as a baseline, so a memory layer must beat it on accuracy or on tokens.")

RESULTS SUMMARY — Notebook 15: agent memory, gpt-6-luna (no reasoning), replay run

LongMemEval-S (n = 500): vector 75.2% [71%-79%] | graph 78.6% [75%-82%] | full 70.8% [67%-75%]
  graph - vector: +3.4% [-0.8%, +7.4%]  not resolvable
  graph - full: +7.8% [+3.0%, +12.6%]  resolvable
  full - vector: -4.4% [-8.4%, -0.4%]  resolvable
  answer-prompt tokens per question: vector 3,197, graph 2,063, full 104,365
  best memory per question type: single-session-user: graph; single-session-assistant: full; single-session-preference: vector; multi-session: graph; knowledge-update: graph; temporal-reasoning: graph; abstention: vector

LoCoMo (n = 1540): vector 66.8% [64%-69%] | graph 73.3% [71%-75%] | full 78.8% [77%-81%]
  graph - vector: +6.6% [+4.2%, +9.0%]  resolvable
  graph - full: -5.5% [-7.6%, -3.4%]  resolvable
  full - vector: +12.0% [+9.7%, +14.4%]  resolvable
  answer-prompt tokens per question: vector 611, graph 1,393, full 20,721
  best memory per question type: multi-hop: full; te